# 1 · Semantic Emoji Picker: many yes/no questions, one request

## The problem

A phone keyboard's emoji search is a **keyword lookup**. Type "pizza" and you get 🍕, but only because someone tagged 🍕
with the word "pizza". Type *"I'm absolutely starving"* and you get nothing: none of those words is a tag.

The usual fixes are a giant synonym table (endless to maintain) or an LLM call that writes back a list of emojis
(slow, costly, and you have to parse free text). What we actually want is a **score for every emoji** that says how well
it fits the *meaning* of what was typed.

## Why Jev

Jev is a **System One** model: it doesn't write text, it returns **typed judgments with calibrated probabilities**,
in about 100 ms, for a fraction of a cent. "Does this emoji fit this text?" is exactly that kind of snap judgment.

The trick is to ask **one independent yes/no question per emoji**, all in a single request. Because the questions don't
compete with each other, several emojis can score high at once: "I'm starving" lifts the **whole food group**, not just
one winner.

## What we'll build

```
"I'm absolutely starving" ──► Jev: 36 Noul questions in ONE call ──► sort by P(yes) ──► keep those above RISE
                               "does 🍕 fit?"  0.9                                       🍕 🍔 ...
                               "does 🐶 fit?"  0.2
                               ...
```

## What you'll learn
- The `Noul` question type: a yes/no question whose answer is a **probability** (`.noul`, 0..1).
- How to ask **dozens of questions in one `system_one` call** with a dict comprehension.
- When to use many independent `Noul`s (multi-select) versus one `Choice` (pick exactly one).
- How a single threshold in code turns probabilities into a feature.

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key from the `.env` file at the repo root (`TYPESAFE_API_KEY`),
- creates the Jev client `jev`. Every judgment in this notebook goes through it,
- imports the Jev question types (`Noul` is the only one this notebook really needs; we borrow `Choice` once for a comparison),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so numbers are easy to eyeball.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder (TYPESAFE_API_KEY)
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=24):
    """A unicode progress bar for a 0..1 value."""
    filled = round(p * width)
    return "█" * filled + "·" * (width - filled)


print("client ready")

client ready


## Step 1 · The vocabulary

Each emoji is an **option**, and its short description is what Jev matches against. There are no tags and no synonym
table: "starving" is never written anywhere below, yet it will match the food emojis. The description just helps Jev
know what each emoji usually *means* (🔥 can mean "hot" or "impressive").

Adding an emoji to the picker is one line.

In [2]:
EMOJIS = {
    "🍕": "pizza, fast food, hungry", "🍔": "burger, fast food",
    "🍣": "sushi, japanese food", "🍎": "apple, fruit, healthy eating",
    "☕": "coffee, caffeine, morning drink", "🍺": "beer, drinks, going out",
    "🐶": "dog, puppy, pet", "🐱": "cat, kitten, pet",
    "🦁": "lion, wild animal, brave", "🐢": "turtle, slow",
    "☀️": "sun, sunny, hot weather", "🌧️": "rain, rainy, wet weather",
    "⛄": "snow, cold, winter", "🌈": "rainbow, colorful, hopeful",
    "😀": "happy, smiling, cheerful", "😢": "sad, crying, upset",
    "😡": "angry, furious, mad", "😍": "love, adoration, crush",
    "😴": "sleepy, tired, bored", "😂": "laughing, hilarious, funny",
    "🎉": "celebration, party, success", "🎂": "birthday, cake",
    "🏆": "trophy, winning, achievement", "🔥": "fire, hot, impressive",
    "✈️": "travel, flight, airplane", "🏖️": "beach, vacation, relax",
    "🚗": "car, driving, road trip", "💻": "coding, computer, programming",
    "📱": "phone, mobile, texting", "📚": "books, studying, reading",
    "🎵": "music, song, listening", "⚽": "soccer, football, sports",
    "🎮": "gaming, video games", "❤️": "love, heart, romance",
    "💰": "money, wealth, rich", "🚀": "rocket, launch, fast, startup",
}
EMOJI_LIST = list(EMOJIS.items())   # a fixed order, so question "e7" always means emoji number 7
print(len(EMOJI_LIST), "emojis")

36 emojis


## Step 2 · Define the questions: one `Noul` per emoji

**New Jev concept: `Noul`.** A `Noul` is a yes/no question. Its answer, `.noul`, is the **probability that the answer
is yes**, from 0 to 1. Because Jev's probabilities are calibrated, 0.9 really means "yes about 9 times out of 10", so you
can put a threshold on it. (You can also pass `criteria=NoulCriteria(true=..., false=...)` to spell out exactly what
counts as yes and no. We don't need that here; notebook 5 uses it.)

We build **one `Noul` per emoji** with a dict comprehension. The dict key (`"e0"`, `"e1"`, ...) is how we look the answer
up later. That's 36 questions, but they will all travel in **one** request.

This cell only **defines** the questions. Nothing is sent to Jev yet.

In [3]:
def emoji_questions():
    """One independent yes/no question per emoji, keyed e0, e1, ..."""
    return {
        f"e{i}": Noul(instructions=f"Does the emoji {e} ({desc}) fit this text?")
        for i, (e, desc) in enumerate(EMOJI_LIST)
    }


EMOJI_QUESTIONS = emoji_questions()
print(len(EMOJI_QUESTIONS), "questions. The first one:")
print(" ", EMOJI_QUESTIONS["e0"])

36 questions. The first one:
  type='noul' instructions='Does the emoji 🍕 (pizza, fast food, hungry) fit this text?' criteria=None


## Step 3 · Ask once and read the raw answer

`jev.system_one(state=..., questions=...)` sends the text (the **state**: what Jev is judging) together with all 36
questions. Every `Noul` answer lands in `r.nouls[key].noul`. We sort them and print the top 10.

In [4]:
r = jev.system_one(state="I'm absolutely starving", questions=EMOJI_QUESTIONS)

ranked = sorted(((EMOJI_LIST[i][0], r.nouls[f"e{i}"].noul) for i in range(len(EMOJI_LIST))),
                key=lambda kv: kv[1], reverse=True)
for e, p in ranked[:10]:
    print(f"  {e}  {bar(p, 20)} {p:.2f}")

total = sum(p for _, p in ranked)
print(f"\nsum of all 36 probabilities: {total:.1f}   (independent questions, so this is NOT 1)")
print("input tokens:", r.usage.input_tokens)

  🍕  ███████████████████· 0.93
  🍔  ████████████████···· 0.80
  🍣  ██████████·········· 0.49
  🍎  ████████············ 0.41
  😢  ██████·············· 0.29
  🐱  █████··············· 0.27
  🦁  █████··············· 0.26
  🐶  █████··············· 0.23
  🔥  ████················ 0.22
  🎂  ████················ 0.18

sum of all 36 probabilities: 6.8   (independent questions, so this is NOT 1)
input tokens: 1143


**What to notice**
- The food emojis are at the top, with 🍕 and 🍔 well above everything else. Nothing in the text says "pizza"; Jev matched on meaning.
- Each emoji got its own probability, and they **don't add up to 1**. Every question was answered on its own.
- Unrelated emojis get low scores, but not exactly zero. That's why we'll need a threshold in code.
- All 36 answers came back from **one** request.

## Step 4 · Why not one `Choice` instead?

**New Jev concept: `Choice`.** A `Choice` asks "which ONE of these labels fits?". Its answer has `.choice` (the winning
label), `.confidence`, and `.probabilities`: a dict with a probability for **every** label, and these **do** add up to 1.

Let's ask the same text as a single `Choice` over all 36 emojis, to see the difference in shape. This is a one-off
comparison; the picker itself uses the `Noul`s.

In [5]:
emoji_choice = Choice(
    instructions="Which single emoji best fits this text?",
    criteria={f"e{i}": f"the emoji {e} ({desc})" for i, (e, desc) in enumerate(EMOJI_LIST)},
)

r = jev.system_one(state="I'm absolutely starving", questions={"best": emoji_choice})
c = r.choices["best"]
print("winner:", EMOJI_LIST[int(c.choice[1:])][0], f"(confidence {c.confidence:.2f})\n")
for key, p in sorted(c.probabilities.items(), key=lambda kv: kv[1], reverse=True)[:5]:
    print(f"  {EMOJI_LIST[int(key[1:])][0]}  {bar(p, 20)} {p:.2f}")
print(f"\nsum of all 36 probabilities: {sum(c.probabilities.values()):.1f}")

winner: 🍕 (confidence 0.94)

  🍕  ███████████████████· 0.94
  🍔  █··················· 0.06
  🍣  ···················· 0.00
  🍎  ···················· 0.00
  ☕  ···················· 0.00

sum of all 36 probabilities: 1.0


**What to notice**
- A `Choice` spreads **one unit** of probability across the labels, so the emojis compete: one winner takes most of it and the other foods are squeezed.
- The independent `Noul`s in Step 3 let 🍕 and 🍔 both score high *at the same time*.
- Rule of thumb: **pick exactly one → `Choice`; "which of these apply?" → one `Noul` each.** A picker that should offer a group is the second kind.

## Step 5 · The picker function, and the threshold that drives it

`suggest()` is the entire "AI" part of the feature: ask the 36 questions, sort by probability. `picks()` is the
**policy**, in plain code:

| Rule | Value | Why |
|---|---|---|
| An emoji "rises" if its probability is above `RISE` | 0.7 | at 0.5 a cheerful sentence lifts a dozen loosely related emojis; 0.7 keeps the strong matches |
| Show at most `MAX_SHOWN` | 6 | a keyboard suggestion strip has limited room |
| If nothing rises, show the single best | | the user always gets *something* |

> **Workshop note:** the original version used 0.5. On happy or celebratory text that let 10+ emojis through
> (the questions are independent, and lots of emojis loosely "fit" a party). Raising the threshold is a one-line
> policy change, with no change to the model or the questions.

The guard at the top of `suggest()` gives a clear message if the kernel was restarted and the cells above weren't re-run.

In [6]:
RISE = 0.7        # an emoji "rises" above this probability
MAX_SHOWN = 6     # at most this many suggestions


def suggest(text, client):
    """Return [(emoji, probability), ...] sorted best first."""
    if "EMOJI_LIST" not in globals():
        raise RuntimeError("EMOJI_LIST is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state=text, questions=emoji_questions())
    scored = [(EMOJI_LIST[i][0], r.nouls[f"e{i}"].noul) for i in range(len(EMOJI_LIST))]
    return sorted(scored, key=lambda kv: kv[1], reverse=True)


def picks(ranked):
    """The policy: everything above RISE, capped at MAX_SHOWN, never empty."""
    risen = [(e, p) for e, p in ranked if p > RISE][:MAX_SHOWN]
    return risen or ranked[:1]


print("suggest() and picks() defined. RISE =", RISE, " MAX_SHOWN =", MAX_SHOWN)

suggest() and picks() defined. RISE = 0.7  MAX_SHOWN = 6


## Step 6 · Run it on examples

Each line is one Jev request (36 questions). We print the emojis that rise, with their probabilities.

In [7]:
QUERIES = [
    "I'm absolutely starving",
    "ugh it's pouring outside",
    "my dog is the best thing ever",
    "so sleepy, need coffee",
    "feeling really down today",
    "booking a trip to the beach",
    "we won the championship!!",
]

for q in QUERIES:
    chosen = picks(suggest(q, jev))
    print(f'{q:<34} -> ' + "  ".join(f"{e} {p:.2f}" for e, p in chosen))

I'm absolutely starving            -> 🍕 0.93  🍔 0.81


ugh it's pouring outside           -> 🌧️ 0.98


my dog is the best thing ever      -> 🐶 0.98  😍 0.93  😀 0.91  ❤️ 0.86  🔥 0.82  🎉 0.72


so sleepy, need coffee             -> ☕ 0.97  😴 0.96


feeling really down today          -> 😢 0.95  🌧️ 0.88


booking a trip to the beach        -> 🏖️ 0.98  ☀️ 0.95  ✈️ 0.90  😀 0.89  📱 0.74


we won the championship!!          -> 🎉 0.98  🏆 0.98  😀 0.95  🔥 0.91  ⚽ 0.86  🍺 0.84


**What to notice**
- Groups rise together: 🍕 and 🍔 for "starving", ☕ and 😴 for "sleepy", sun and travel for the beach, a whole party for the championship.
- Mood and topic both count: "my dog is the best thing ever" lifts 🐶 *and* the happy/love faces.
- Short, single-mood texts give only one or two picks ("pouring outside" is basically just 🌧️), while big happy texts hit the `MAX_SHOWN` cap of 6. Both are exactly what the threshold and cap in Step 5 say.
- Now and then a looser association slips over the line (for example 📱 for *booking* a trip). Tuning `RISE` or the emoji descriptions is how you deal with that.

## Step 7 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired up with `ipywidgets`, and an HTML
renderer. The picker logic above (`suggest()` and `picks()`) is unchanged.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

# small style kit
INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def text_app(placeholder, render, button="Run", value=""):
    """A text box + button. On submit, call render(text) and show the HTML it returns."""
    inp = widgets.Text(value=value, placeholder=placeholder, continuous_update=False,
                       layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>thinking...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # fires when you press Enter
    display(widgets.HBox([inp, btn]), out)

The renderer shows **every** emoji above 0.2, and scales **size and opacity with the probability**, so the strong
matches literally grow and the weak ones fade. Hover an emoji to see its probability. The emojis that would be picked
(`picks()`) get a small underline.

In [9]:
def emoji_html(text):
    ranked = suggest(text, jev)
    chosen = {e for e, _ in picks(ranked)}
    chips = ""
    for e, p in ranked:
        if p < 0.2:
            continue
        size = 18 + p * 46          # bigger = more likely
        opacity = 0.35 + 0.65 * p
        underline = f"border-bottom:3px solid {CORAL};" if e in chosen else ""
        chips += (f'<span style="font-size:{size:.0f}px;opacity:{opacity:.2f};margin:0 6px;{underline}" '
                  f'title="{p:.2f}">{e}</span>')
    return box(f'<div style="{CARD}"><div style="color:{SLATE};font-size:13px;">you typed: <b>{text}</b></div>'
               f'<div style="line-height:1.9;">{chips}</div></div>')


display(HTML("".join(emoji_html(q) for q in ["I'm absolutely starving", "my dog is the best thing ever"])))

## Step 8 · Try it yourself

Type into the box and press **Enter** or click the button (needs a live kernel). Good ones to try in a workshop:
- `I just got promoted!` → celebration and trophy emojis should rise together (🎉 🏆 😀 ...).
- `stuck in traffic again` → expect 🚗 and 🐢 (slow!) near the top, with annoyed or sad faces.
- `freezing cold morning` → ⛄ should win clearly. Nothing says "snow": Jev reads the *scene*, not keywords.
- `lol that meme` → 😂 should win; a good check that it picks up slang.

In [10]:
text_app("type a feeling or thing, e.g. 'I'm so tired'", emoji_html, "Suggest")

Output()

## Recap and exercises

**Key ideas**
1. **One `Noul` per option** turns any option list into a ranked, multi-select semantic match, in a single request.
2. `Noul`s are independent (they don't sum to 1), so groups can rise together. A `Choice` makes options compete for one winner.
3. **The model measures, the code decides.** Jev returned 36 probabilities; `RISE` and `MAX_SHOWN` are ordinary Python you can tune.
4. The option descriptions are the model's whole world: a better description is a better match.

**Exercises**
- Change `RISE` to 0.5 and re-run Step 6. Which texts suddenly suggest too much?
- Add five emojis of your own (e.g. 🎸, 🧘, 🤒) to `EMOJIS` and try texts that should match them.
- Change the description of 🔥 to just `"fire, flames"`. Does it stop rising for "we won the championship!!"?
- Swap the domain: replace emojis with **product tags** or **support labels**. The code doesn't change at all.